# Edge-Cloud Streaming Pipeline — v7 Results Visualization

**Judul:** Strategi Arsitektur Edge-Cloud Berbasis Fusi Data Multimodal pada Ekosistem Digital Twin Web-3D untuk Prediksi Energi Bangunan Cerdas

---

## Ringkasan Eksperimen
- Dataset: 2,027,520 records (90 hari sensor IoT)
- Model: Ridge Regression α=1e-2, 17 fitur
- Anomaly threshold: z=2.5
- Anomali disuntikkan: 200 hard + 2,000 soft
- Latensi edge median: 1.3 ms
- Latensi cloud: 275 ms (network + processing + sync)

---

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Load streaming results
with open('streaming_results.pkl', 'rb') as f:
    results = pickle.load(f)

df = pd.DataFrame([
    {
        'sample_idx': r.sample_idx,
        'timestamp': r.timestamp,
        'anomaly': r.anomaly,
        'routed_to_cloud': r.routed_to_cloud,
        'edge_latency_ms': r.edge_latency_ms,
        'cloud_latency_ms': r.cloud_latency_ms,
        'total_latency_ms': r.total_latency_ms,
        'energy_mw': r.energy_mw,
        'energy_score': r.energy_score,
        'daya': r.daya,
        'pred_daya': r.pred_daya,
    }
    for r in results
])

df['timestamp_dt'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp_dt'].dt.hour
df['day_of_week'] = df['timestamp_dt'].dt.dayofweek
df['day'] = df['timestamp_dt'].dt.day

total = len(df)
n_anomaly = df['anomaly'].sum()
n_cloud = df['routed_to_cloud'].sum()

print(f'Loaded: {total:,} records')
print(f'Anomalies: {n_anomaly:,} ({n_anomaly/total*100:.2f}%)')
print(f'Cloud-routed: {n_cloud:,}')
print(f'Edge latency: mean={df["edge_latency_ms"].mean():.2f} ms')
print(f'Energy: mean={df["energy_mw"].mean():.2f} mW, std={df["energy_mw"].std():.2f} mW')
print(f'Daya: mean={df["daya"].mean():.2f}W, std={df["daya"].std():.2f}W')
print(f'Prediction accuracy: MAE={abs(df["daya"] - df["pred_daya"]).mean():.4f}W')

In [ ]:
# ============================================================
# FIGURE 1: Full timeline — Daya + Pred + Anomaly overlay
# ============================================================
# Subsample for clarity (show every 10th record)
sub = df.iloc[::10].copy()

fig = plt.figure(figsize=(16, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.3)

ax1 = fig.add_subplot(gs[0])

# Normal records
normal = sub[~sub['anomaly']]
anom = sub[sub['anomaly']]

ax1.plot(normal['timestamp_dt'], normal['daya'], '.', markersize=2, alpha=0.3, color='steelblue', label='Normal (clean)', zorder=2)
ax1.plot(anom['timestamp_dt'], anom['daya'], '^', markersize=4, color='red', alpha=0.7, label=f'Anomaly ({len(anom):,})', zorder=3)
ax1.plot(sub['timestamp_dt'], sub['pred_daya'], '--', linewidth=0.8, color='darkgreen', alpha=0.6, label='Ridge Prediction', zorder=1)

ax1.set_xlabel('Time', fontsize=11)
ax1.set_ylabel('Daya (W)', fontsize=11)
ax1.set_title('Time Series: Daya Actual vs Ridge Prediction (Subsample 10x)', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.tick_params(axis='x', rotation=30)
ax1.axhline(y=df['daya'].mean(), color='gray', linestyle=':', alpha=0.5, label=f'Mean {df["daya"].mean():.1f}W')

# Anomaly rate per 1000-record window
ax2 = fig.add_subplot(gs[1])
window = 1000
n_windows = len(df) // window
anomaly_rates = [df.iloc[i*window:(i+1)*window]['anomaly'].mean() for i in range(n_windows)]
windows = np.arange(n_windows) * window

ax2.bar(windows, [r*100 for r in anomaly_rates], width=window*0.8, color='orangered', alpha=0.6, label='Anomaly Rate')
ax2.set_xlabel('Window (records)', fontsize=11)
ax2.set_ylabel('Anomaly Rate (%)', fontsize=11)
ax2.set_title(f'Anomaly Detection Rate per {window:,}-Record Window', fontsize=12)
ax2.axhline(y=n_anomaly/total*100, color='red', linestyle='--', linewidth=2, label=f'Global Mean {n_anomaly/total*100:.2f}%')
ax2.legend(fontsize=9)
ax2.tick_params(axis='x', rotation=30)

plt.savefig('v7_figure1_timeline.png', bbox_inches='tight')
print('Saved: v7_figure1_timeline.png')
plt.close()

In [ ]:
# ============================================================
# FIGURE 2: Edge vs Cloud Latency Distribution
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Edge latency distribution
axes[0].hist(df['edge_latency_ms'], bins=100, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.3)
axes[0].set_xlabel('Edge Latency (ms)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title(f'Edge Latency Distribution\nMean={df["edge_latency_ms"].mean():.2f} ms, P99={np.percentile(df["edge_latency_ms"], 99):.2f} ms', fontsize=12)
axes[0].axvline(x=2.0, color='red', linestyle='--', linewidth=2, label='SLA (<2ms)')
axes[0].legend(fontsize=9)

# Panel B: Cloud latency (only for anomalies)
anom_df = df[df['anomaly']].copy()
if len(anom_df) > 0:
    axes[1].hist(anom_df['cloud_latency_ms'], bins=50, color='coral', alpha=0.7, edgecolor='black', linewidth=0.3)
    axes[1].set_xlabel('Cloud Latency (ms)', fontsize=11)
    axes[1].set_ylabel('Frequency', fontsize=11)
    axes[1].set_title(f'Cloud Latency (Anomaly Only)\nMean={anom_df["cloud_latency_ms"].mean():.1f} ms', fontsize=12)
    axes[1].axvline(x=275, color='navy', linestyle='--', linewidth=2, label='Config Target')
    axes[1].legend(fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No anomalies', ha='center', va='center', fontsize=14)

# Panel C: Total latency breakdown
total_edge = df[~df['anomaly']]['edge_latency_ms']
total_cloud = df[df['anomaly']]['total_latency_ms']

x_pos = [0, 1]
medians = [total_edge.median(), total_cloud.median()]
means = [total_edge.mean(), total_cloud.mean()]
colors_box = ['steelblue', 'coral']
labels = ['Normal\n(Edge Only)', 'Anomaly\n(Edge + Cloud)']

bp = axes[2].boxplot([total_edge.tolist()[:10000], total_cloud.tolist()[:10000]], positions=[0.5, 1.5], widths=0.6, 
                       patch_artist=True, boxprops=dict(linewidth=1.5))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[2].scatter([0.5], [means[0]], marker='D', s=100, c='gold', edgecolor='black', zorder=5, label='Mean')
axes[2].scatter([1.5], [means[1]], marker='D', s=100, c='gold', edgecolor='black', zorder=5)
axes[2].set_xticks([0.5, 1.5])
axes[2].set_xticklabels(labels, fontsize=10)
axes[2].set_ylabel('Total Latency (ms)', fontsize=11)
axes[2].set_title('Latency Breakdown: Normal vs Anomaly Flow', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].set_ylim(0, 350)

plt.suptitle('Edge-Cloud Latency Analysis (v7 Streaming Pipeline)', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('v7_figure2_latency.png', bbox_inches='tight', dpi=150)
print('Saved: v7_figure2_latency.png')
plt.close()

In [ ]:
# ============================================================
# FIGURE 3: Energy Consumption Analysis
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Energy per record
normal_eng = df[~df['anomaly']]['energy_mw']
anom_eng = df[df['anomaly']]['energy_mw']

bp = axes[0].boxplot([normal_eng.tolist()[:10000], anom_eng.tolist()[:10000]], positions=[0.5, 1.5], widths=0.6,
                      patch_artist=True, boxprops=dict(linewidth=1.5))
for patch, color in zip(bp['boxes'], ['steelblue', 'coral']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_xticks([0.5, 1.5])
axes[0].set_xticklabels(['Normal', 'Anomaly'], fontsize=10)
axes[0].set_ylabel('Energy (mW)', fontsize=11)
axes[0].set_title(f'Energy per Record\nNormal={normal_eng.median():.1f} mW | Anomaly={anom_eng.median():.1f} mW', fontsize=12)

# Panel B: Cumulative energy over time
window = 10000
n_windows = min(202, len(df) // window)
cum_edge = []
cum_cloud = []
for i in range(n_windows):
    chunk = df.iloc[i*window:(i+1)*window]
    cum_edge.append(chunk[~chunk['anomaly']]['energy_mw'].sum())
    cum_cloud.append(chunk[chunk['anomaly']]['energy_mw'].sum())

x = np.arange(n_windows) + 1
axes[1].bar(x - 0.25, cum_edge, 0.5, color='steelblue', alpha=0.7, label='Edge (normal)')
axes[1].bar(x + 0.25, cum_cloud, 0.5, color='coral', alpha=0.7, label='Cloud (anomaly)')
axes[1].set_xlabel('10K Record Windows', fontsize=11)
axes[1].set_ylabel('Total Energy (mW)', fontsize=11)
axes[1].set_title(f'Cumulative Energy per 10K Records\nTotal Edge={sum(cum_edge):.0f} | Cloud={sum(cum_cloud):.0f}', fontsize=12)
axes[1].legend(fontsize=9)

# Panel C: Energy score vs Daya relationship
axes[2].scatter(df['daya'], df['energy_score'], s=1, alpha=0.3, color='purple', label='All records')
anom_sub = df[df['anomaly']]
axes[2].scatter(anom_sub['daya'], anom_sub['energy_score'], s=8, color='red', alpha=0.6, label=f'Anomaly ({len(anom_sub):,})', zorder=5)
axes[2].set_xlabel('Daya (W)', fontsize=11)
axes[2].set_ylabel('Energy Score', fontsize=11)
axes[2].set_title('Energy Score vs Power Consumption\nRed dots = Detected anomalies', fontsize=12)
axes[2].legend(fontsize=9)

plt.suptitle('Energy Consumption Analysis (v7 Streaming Pipeline)', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('v7_figure3_energy.png', bbox_inches='tight', dpi=150)
print('Saved: v7_figure3_energy.png')
plt.close()

In [ ]:
# ============================================================
# FIGURE 4: Prediction Accuracy Deep Dive
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel A: Actual vs Predicted scatter (subsampled)
sub_sample = df.iloc[::100]
axes[0, 0].scatter(sub_sample['daya'], sub_sample['pred_daya'], s=3, alpha=0.3, color='steelblue')
min_val = min(sub_sample['daya'].min(), sub_sample['pred_daya'].min())
max_val = max(sub_sample['daya'].max(), sub_sample['pred_daya'].max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction (y=x)')
axes[0, 0].set_xlabel('Actual Daya (W)', fontsize=11)
axes[0, 0].set_ylabel('Predicted Daya (W)', fontsize=11)
axes[0, 0].set_title('Actual vs Predicted (Subsample 100x)\nMAE = {:.4f}W'.format(abs(sub_sample['daya'] - sub_sample['pred_daya']).mean()), fontsize=12)
axes[0, 0].legend(fontsize=9)
axes[0, 0].axhline(y=df['daya'].mean(), color='orange', linestyle=':', alpha=0.5, label=f'Mean {df["daya"].mean():.1f}W')
axes[0, 0].axvline(x=df['daya'].mean(), color='orange', linestyle=':', alpha=0.5)
axes[0, 0].legend(fontsize=9)

# Panel B: Prediction error distribution
errors = df['daya'] - df['pred_daya']
axes[0, 1].hist(errors, bins=200, color='teal', alpha=0.7, edgecolor='black', linewidth=0.3)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0, 1].axvline(x=errors.median(), color='orange', linestyle=':', linewidth=2, label=f'Median={errors.median():.4f}')
axes[0, 1].set_xlabel('Prediction Error (W)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title(f'Prediction Error Distribution\nMean={errors.mean():.4f} | Std={errors.std():.4f} | IQR=[{errors.quantile(0.25):.4f}, {errors.quantile(0.75):.4f}]', fontsize=11)
axes[0, 1].legend(fontsize=8)

# Panel C: Error by hour of day
hourly_err = df.groupby('hour')['daya'].apply(lambda x: abs(x - df.loc[x.index, 'pred_daya']).mean()).reset_index()
hourly_err.columns = ['hour', 'mean_absolute_error']
colors_hour = ['green' if 6 <= h <= 20 else 'steelblue' for h in hourly_err['hour']]
bars = axes[1, 0].bar(hourly_err['hour'], hourly_err['mean_absolute_error'], color=colors_hour, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1, 0].set_xlabel('Hour of Day', fontsize=11)
axes[1, 0].set_ylabel('Mean Abs Error (W)', fontsize=11)
axes[1, 0].set_title('Prediction Error by Hour\nGreen=Daytime (business hours) | Blue=Night', fontsize=11)
axes[1, 0].set_xticks(range(0, 24, 2))
for bar, val in zip(bars, hourly_err['mean_absolute_error']):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f'{val:.3f}', ha='center', va='bottom', fontsize=6)

# Panel D: Error cumulative over time
rolling_error = errors.rolling(10000, min_periods=1).mean()
axes[1, 1].plot(rolling_error.values, linewidth=0.5, color='darkred', alpha=0.7)
axes[1, 1].fill_between(range(len(rolling_error)), rolling_error.values - errors.std(), rolling_error.values + errors.std(), alpha=0.2, color='salmon')
axes[1, 1].set_xlabel('Window (records)', fontsize=11)
axes[1, 1].set_ylabel('Cumulative MAE (W)', fontsize=11)
axes[1, 1].set_title(f'Running MAE over 10K windows\nStabilized at {rolling_error.values[-1]:.4f}W | ±{errors.std():.4f}', fontsize=11)

plt.suptitle('Prediction Accuracy Deep Dive (v7 Ridge Model)', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('v7_figure4_accuracy.png', bbox_inches='tight', dpi=150)
print('Saved: v7_figure4_accuracy.png')
plt.close()

In [ ]:
# ============================================================
# FIGURE 5: Architecture Decision Flow — Sankey-style bar chart
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Processing flow
normal_pct = (1 - n_anomaly/total) * 100
anom_pct = n_anomaly / total * 100

labels_flow = ['2,027,520\nTotal Records', f'{normal_pct:.1f}%\nProcessed on Edge', f'{anom_pct:.1f}%\nRouted to Cloud']
values_flow = [100, normal_pct, anom_pct]
colors_flow = ['steelblue', 'lightgreen', 'coral']

bars_a = axes[0].barh(labels_flow, values_flow, color=colors_flow, edgecolor='black', linewidth=1.5, height=0.5)
for bar, val in zip(bars_a, values_flow):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', va='center', fontweight='bold', fontsize=10)

axes[0].set_xlim(0, 110)
axes[0].set_title('Processing Flow Decision', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Percentage (%)', fontsize=11)

# Panel B: Resource utilization comparison
metrics_labels = ['Avg Latency\n(ms)', 'Avg Energy\n(mW)', 'Throughput\n(rec/s)']
edge_only_vals = [df[~df['anomaly']]['edge_latency_ms'].mean(), df[~df['anomaly']]['energy_mw'].mean(), 100]
full_flow_vals = [df[df['anomaly']]['total_latency_ms'].mean(), df[df['anomaly']]['energy_mw'].mean(), 100]

x = np.arange(3)
width = 0.35
bars1 = axes[1].bar(x - width/2, edge_only_vals, width, label='Normal (Edge Only)', color='steelblue', alpha=0.8, edgecolor='black')
bars2 = axes[1].bar(x + width/2, full_flow_vals, width, label='Anomaly (Edge+Cloud)', color='coral', alpha=0.8, edgecolor='black')

axes[1].set_ylabel('Relative Value (normalized to edge=100 for throughput)', fontsize=10)
axes[1].set_title('Resource Comparison: Edge-Only vs Edge+Cloud', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_labels, fontsize=9)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('v7_figure5_architecture.png', bbox_inches='tight', dpi=150)
print('Saved: v7_figure5_architecture.png')
plt.close()

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================
print('=' * 60)
print('V7 STREAMING PIPELINE — RESULTS SUMMARY')
print('=' * 60)
print(f'Dataset: {total:,} records ({len(df["timestamp_dt"].unique()):,} unique timestamps)')
print(f'\n--- Anomaly Detection ---')
print(f'  Threshold: z=2.5')
print(f'  Injected: 200 hard + 2,000 soft = 2,200 total')
print(f'  Detected: {n_anomaly:,} ({n_anomaly/total*100:.2f}%)')
print(f'  Cloud routed: {n_cloud:,} ({n_cloud/total*100:.2f}%)')
print(f'\n--- Latency ---')
print(f'  Edge (normal): mean={df[~df["anomaly"]]["edge_latency_ms"].mean():.2f} ms, P95={np.percentile(df[~df["anomaly"]]["edge_latency_ms"], 95):.2f} ms')
print(f'  Cloud (anomaly): mean={anom_df["cloud_latency_ms"].mean():.1f} ms, P95={np.percentile(anom_df["cloud_latency_ms"], 95):.1f} ms')
print(f'  Total anomaly flow: mean={df[df["anomaly"]]["total_latency_ms"].mean():.1f} ms')
print(f'\n--- Energy ---')
print(f'  Edge (normal): mean={df[~df["anomaly"]]["energy_mw"].mean():.2f} mW')
print(f'  Cloud (anomaly): mean={anom_df["energy_mw"].mean():.2f} mW')
print(f'  Overall avg: {df["energy_mw"].mean():.2f} mW')
total_energy = df['energy_mw'].sum()
print(f'  Total energy consumed: {total_energy:,.0f} mW ({total_energy/1000:.1f} Wh)')
print(f'\n--- Prediction Accuracy ---')
mae = abs(df['daya'] - df['pred_daya']).mean()
rmse = np.sqrt(((df['daya'] - df['pred_daya'])**2).mean())
ape = abs((df['daya'] - df['pred_daya']) / df['daya']).mean() * 100
print(f'  MAE: {mae:.4f} W')
print(f'  RMSE: {rmse:.4f} W')
print(f'  MAPE: {ape:.2f} %')
print(f'  Mean pred: {df["pred_daya"].mean():.2f} W')
print(f'  Mean actual: {df["daya"].mean():.2f} W')
print(f'\n--- Throughput ---')
print(f'  Avg: {total/920:.0f} records/second')
print(f'  Total time: ~920 seconds')
print('=' * 60)
print('All figures saved as PNG files in working directory.')
print('Figures: v7_figure1_timeline.png, v7_figure2_latency.png, v7_figure3_energy.png,')
print('         v7_figure4_accuracy.png, v7_figure5_architecture.png')
